In [3]:
!pip install torch
!pip install numpy
!pip install open_clip-torch
!pip install Pillow
!pip install scikit-learn
!pip install torch pandas

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import torch
import numpy as np
import random
import open_clip
from PIL import Image
import pandas as pd
import glob

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

model_id = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading BiomedCLIP model on {device}...")

model, _, preprocess_val = open_clip.create_model_and_transforms(model_id)
tokenizer = open_clip.get_tokenizer(model_id)
model.eval()
model.to(device)

colour_labels = ["a T wave with a high amount of blue/green", "a T wave with a high amount of orange/red"]
morphology_label_sets = {"No Label": ["Normal", "Abnormal"], "L1": ["Normal T wave", "Abnormal T wave"], "L2": ["Normal", "Flat", "Large", "Inverted", "Biphasic", "Notched"], "L3": ["Normal T wave", "Abnormal T wave with flat amplitude", "Abnormal large T wave, taller than initial signal", "''Abnormal large T wave, broader and larger than initial signal''', "Abnormal T wave with a minor inversion", "Abnormal T wave with a deep inversion", "Abnormal T wave with a giant inversion", "Abnormal Biphasic T wave(Type A)", "Abnormal Biphasic T wave(Type B)", "Abnormal Bifid or Notched T wave"]}

text_c = tokenizer(colour_labels).to(device)
with torch.no_grad():
    text_features_c = model.encode_text(text_c)
    text_features_c /= text_features_c.norm(dim=-1, keepdim=True)

def evaluate_image(image_path, text_features_m, text_features_c, morphology_labels, alpha=0.5):
    raw_image = Image.open(image_path).convert('RGB')
    image = preprocess_val(raw_image).unsqueeze(0).to(device)
    with torch.no_grad():
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        logit_scale = model.logit_scale.exp()
        logits_per_image_m = logit_scale * image_features @ text_features_m.t()
        logits_per_image_c = logit_scale * image_features @ text_features_c.t()
        probs_m = logits_per_image_m.softmax(dim=-1).cpu().numpy()[0]
        probs_c = logits_per_image_c.softmax(dim=-1).cpu().numpy()[0]
        p_blue_green = probs_c[0]
        p_orange_red = probs_c[1]
        weight_blue_green = (p_blue_green * alpha) + (1.0 - alpha)
        weight_orange_red = (p_orange_red * alpha) + (1.0 - alpha)
        mixed_probs_m = []
        for i, label in enumerate(morphology_labels):
            weight = weight_blue_green if "Normal" in label else weight_orange_red
            mixed_probs_m.append(probs_m[i] * weight)
        mixed_probs_m = np.array(mixed_probs_m)
        mixed_probs_m /= mixed_probs_m.sum()
        none_probs = np.ones(len(morphology_labels)) / len(morphology_labels)
    return probs_m, probs_c, mixed_probs_m, none_probs

Loading BiomedCLIP model on cuda...


In [6]:
import glob
import numpy as np

high_risk_dir = "/content/drive/MyDrive/HeartBeat_Coloured/All/At risk/*.png"
no_risk_dir = "/content/drive/MyDrive/HeartBeat_Coloured/All/No risk/*.png"
image_paths_abnormal = np.array(sorted(glob.glob(high_risk_dir)))
image_paths_normal = np.array(sorted(glob.glob(no_risk_dir)))
ground_truth_abnormal = np.array(['Abnormal'] * len(image_paths_abnormal))
ground_truth_normal = np.array(['Normal'] * len(image_paths_normal))
image_paths = np.concatenate((image_paths_abnormal, image_paths_normal))
ground_truth = np.concatenate((ground_truth_abnormal, ground_truth_normal))
print(f"Found {len(image_paths_abnormal)} images.")

Found 181 high-risk (abnormal) images.


In [7]:
import pandas as pd
import numpy as np
import torch
import os

desired_label_set_order = ["No Label", "L1", "L2", "L3"]
real_inference_results_colour = []
for label_set_name in desired_label_set_order:
    morph_labels = morphology_label_sets[label_set_name]
    text_m = tokenizer(morph_labels).to(device)
    with torch.no_grad():
        text_features_m = model.encode_text(text_m)
        text_features_m /= text_features_m.norm(dim=-1, keepdim=True)
    for path, true_label in zip(image_paths, ground_truth):
        try:
            p_morph, p_col, p_both, p_none = evaluate_image(path, text_features_m, text_features_c, morph_labels, alpha=0.5)
            pred_morph = morph_labels[np.argmax(p_morph)]
            pred_both = morph_labels[np.argmax(p_both)]
            pred_none = morph_labels[np.argmax(p_none)]
            pred_col = 'Normal' if np.argmax(p_col) == 0 else 'Abnormal'
            file_name = os.path.basename(path)
            real_inference_results_colour.append({'File_Name': file_name, 'True_Label': true_label, 'Predicted_Label': pred_morph, 'Strategy': 'No Colour + Morphology Context', 'Label_Set': label_set_name})
            real_inference_results_colour.append({'File_Name': file_name, 'True_Label': true_label, 'Predicted_Label': pred_both, 'Strategy': 'Colour + Morphology Context', 'Label_Set': label_set_name})
            real_inference_results_colour.append({'File_Name': file_name, 'True_Label': true_label, 'Predicted_Label': pred_none, 'Strategy': 'No Colour + No Morphology Context', 'Label_Set': label_set_name})
            real_inference_results_colour.append({'File_Name': file_name, 'True_Label': true_label, 'Predicted_Label': pred_col, 'Strategy': 'Colour + No Morphology Context', 'Label_Set': label_set_name})
        except Exception as e: print(f"Error: {e}")
df_real_inference_colour = pd.DataFrame(real_inference_results_colour)

Starting inference loop...


In [8]:
import pandas as pd
import numpy as np
import math
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

def map_to_binary(label):
    if 'normal' in label.lower() and 'abnormal' not in label.lower(): return 'Normal'
    elif 'normal' in label.lower() and label.lower().startswith('normal'): return 'Normal'
    else: return 'Abnormal'

df_real_inference_colour['Binary_Prediction'] = df_real_inference_colour['Predicted_Label'].apply(map_to_binary)
df_real_inference_colour['True_Binary_Num'] = (df_real_inference_colour['True_Label'] == 'Abnormal').astype(int)
df_real_inference_colour['Pred_Binary_Num'] = (df_real_inference_colour['Binary_Prediction'] == 'Abnormal').astype(int)
metrics_colour_results = []
grouped_colour = df_real_inference_colour.groupby(['Label_Set', 'Strategy'])
for (label_set, strategy), group in grouped_colour:
    y_true, y_pred = group['True_Binary_Num'], group['Pred_Binary_Num']
    TP = len(group[(group['True_Label'] == 'Abnormal') & (group['Binary_Prediction'] == 'Abnormal')])
    FN = len(group[(group['True_Label'] == 'Abnormal') & (group['Binary_Prediction'] == 'Normal')])
    TN = len(group[(group['True_Label'] == 'Normal') & (group['Binary_Prediction'] == 'Normal')])
    FP = len(group[(group['True_Label'] == 'Normal') & (group['Binary_Prediction'] == 'Abnormal')])
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1 = 2 * (ppv * sensitivity) / (ppv + sensitivity) if (ppv + sensitivity) > 0 else 0.0
    numerator_mcc = (TP * TN) - (FP * FN)
    denominator_product = (TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)
    mcc = numerator_mcc / math.sqrt(denominator_product) if denominator_product > 0 else 0.0
    try: roc_auc = roc_auc_score(y_true, y_pred)
    except: roc_auc = 0.0
    metrics_colour_results.append({'Label_Set': label_set, 'Strategy': strategy, 'TP': TP, 'FN': FN, 'TN': TN, 'FP': FP, 'Sensitivity': sensitivity, 'Specificity': specificity, 'PPV': ppv, 'F1_Score': f1, 'MCC': mcc, 'ROC_AUC': roc_auc})
df_metrics_colour = pd.DataFrame(metrics_colour_results)
desired_order = ['No Label', 'L1', 'L2', 'L3']
df_metrics_colour['Label_Set'] = pd.Categorical(df_metrics_colour['Label_Set'], categories=desired_order, ordered=True)
df_metrics_colour = df_metrics_colour.sort_values('Label_Set')
display(df_metrics_colour)

,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
12,No Label,Colour + Morphology Context,159,22,850,4004,0.878453,0.175113,0.038194,0.073204,0.026352,0.526783
